# Surface Water Temporal Change Analysis
### Description:
This Python script performs multi-temporal surface water analysis for Northern Manitoba using Sentinel-2 and Landsat 8/9 imagery. It computes NDWI-based water masks for summer months (June–August) across the years 2016–2021, generating both year-to-year and full period (earliest → latest year) change analyses. Outputs include per-sensor ΔNDWI maps, per-pixel % change maps, multi-year persistence maps, quantitative CSV summaries, and multi-panel overview figures for rapid visualization. Thresholds for binary water masks (ndwi_threshold) and minimal change detection (delta_threshold) are configurable. The script is designed to provide both visual and statistical insights into surface water dynamics over time.

### Inputs 
- NDWI rasters and water masks per sensor/year

### Outputs
1. Year-to-Year ΔNDWI maps
2. Year-to-Year water gain/loss overlays
3. Multi-year water persistence maps
4. Earliest → latest year ΔNDWI maps
5. Earliest → latest per-pixel % NDWI change maps
6. Quantitative CSV summary
7. Sensor-specific multi-panel summary figures
8. Optional per-pixel % change for consecutive years

In [5]:
import os
import glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import pandas as pd

In [6]:
# -----------------------------
# Configuration
# -----------------------------
OUTPUT_DIR = "../data/results"
INPUT_DIR = "../data/raw"
sensors = ["landsat8", "sentinel2"]
expected_years = ["2016", "2017", "2018", "2019", "2020", "2021"]

ndwi_threshold = 0.3         # binary water mask threshold
delta_threshold = 0.05       # NDWI change threshold

FILES = {}
summary_rows = []

In [7]:
# -----------------------------
# Build file dictionary
# -----------------------------
print(f"\n=== Files for sensor: {sensor} ===")
for sensor in sensors:
    sensor_folder = os.path.join(INPUT_DIR, sensor)
    FILES[sensor] = {"ndwi": {}, "mask": {}}

    print("NDWI files:")    
    for f in glob.glob(os.path.join(sensor_folder, f"{sensor}_ndwi_*.tif")):
        year = os.path.basename(f).split("_")[-1].split(".")[0]
        FILES[sensor]["ndwi"][year] = f
        print(f"  {year}: {f}")
        

    print("Water mask files:")
    for f in glob.glob(os.path.join(sensor_folder, f"{sensor}_watermask_*.tif")):
        year = os.path.basename(f).split("_")[-1].split(".")[0]
        FILES[sensor]["mask"][year] = f
        print(f"  {year}: {f}")

    # Warn if missing
    for year in expected_years:
        if year not in FILES[sensor]["ndwi"]:
            print(f"WARNING: NDWI missing for {sensor}, year {year}")
        if year not in FILES[sensor]["mask"]:
            print(f"WARNING: Water mask missing for {sensor}, year {year}")


=== Files for sensor: sentinel2 ===
NDWI files:
  2016: ../data/raw/landsat8/landsat8_ndwi_2016.tif
  2017: ../data/raw/landsat8/landsat8_ndwi_2017.tif
  2019: ../data/raw/landsat8/landsat8_ndwi_2019.tif
  2018: ../data/raw/landsat8/landsat8_ndwi_2018.tif
  2020: ../data/raw/landsat8/landsat8_ndwi_2020.tif
  2021: ../data/raw/landsat8/landsat8_ndwi_2021.tif
Water mask files:
  2021: ../data/raw/landsat8/landsat8_watermask_2021.tif
  2020: ../data/raw/landsat8/landsat8_watermask_2020.tif
  2018: ../data/raw/landsat8/landsat8_watermask_2018.tif
  2019: ../data/raw/landsat8/landsat8_watermask_2019.tif
  2017: ../data/raw/landsat8/landsat8_watermask_2017.tif
  2016: ../data/raw/landsat8/landsat8_watermask_2016.tif
NDWI files:
  2021: ../data/raw/sentinel2/sentinel2_ndwi_2021.tif
  2020: ../data/raw/sentinel2/sentinel2_ndwi_2020.tif
  2018: ../data/raw/sentinel2/sentinel2_ndwi_2018.tif
  2019: ../data/raw/sentinel2/sentinel2_ndwi_2019.tif
  2017: ../data/raw/sentinel2/sentinel2_ndwi_2017.t

In [8]:
# -----------------------------
# Analysis Loop
# -----------------------------
for sensor in sensors:
    print(f"\n=== Processing sensor: {sensor} ===")
    sensor_folder = os.path.join(OUTPUT_DIR, sensor)
    os.makedirs(sensor_folder, exist_ok=True)

    # Load NDWI stack
    ndwi_stack = []
    years_available = sorted(FILES[sensor]["ndwi"].keys())
    for year in years_available:
        with rasterio.open(FILES[sensor]["ndwi"][year]) as src:
            ndwi_stack.append(src.read(1))
            transform = src.transform
            crs = src.crs
            shape = src.shape
    ndwi_stack = np.array(ndwi_stack)  #shape: (years, rows, cols)
    # -----------------------------
    # NDWI Threshold Suggestion
    # -----------------------------
    ndwi_flat = ndwi_stack.flatten()
    
    # Compute suggested threshold (median NDWI) and percentiles
    suggested_threshold = np.percentile(ndwi_flat, 50)       # median
    lower_water_threshold = np.percentile(ndwi_flat, 5)      # very wet pixels
    upper_water_threshold = np.percentile(ndwi_flat, 95)     # mostly water areas

    print(f"{sensor} NDWI threshold suggestions:")
    print(f"  Median NDWI = {suggested_threshold:.2f}")
    print(f"  5th percentile (wet pixels) = {lower_water_threshold:.2f}")
    print(f"  95th percentile = {upper_water_threshold:.2f}")

    # Use median as dynamic threshold for binary water mask
    ndwi_threshold = suggested_threshold
    
    # -----------------------------
    # Multi-year water persistence map
    # -----------------------------
    binary_stack = ndwi_stack > ndwi_threshold
    persistence = np.sum(binary_stack, axis=0)

    plt.figure(figsize=(8,6))
    plt.imshow(persistence, cmap='viridis')
    plt.colorbar(label="Years with Water")
    plt.title(f"{sensor} Multi-Year Water Persistence")
    out_file = os.path.join(sensor_folder, f"{sensor}_water_persistence.png")
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Multi-year water persistence map saved: {out_file}")

    # -----------------------------
    # Year-to-Year ΔNDWI, gain/loss, and per-pixel % change
    # -----------------------------
    pixel_area_km2 = 0.0009 if sensor=="landsat8" else 0.0001
    for i in range(1, len(years_available)):
        year_prev = years_available[i-1]
        year_curr = years_available[i]
        delta = ndwi_stack[i] - ndwi_stack[i-1]
        gain = delta > delta_threshold
        loss = delta < -delta_threshold
        area_gain = np.sum(gain) * pixel_area_km2
        area_loss = np.sum(loss) * pixel_area_km2
        net_change = area_gain - area_loss

        summary_rows.append({
            "Sensor": sensor,
            "Year_Prev": year_prev,
            "Year_Curr": year_curr,
            "Area_Gained_km2": area_gain,
            "Area_Lost_km2": area_loss,
            "Net_Change_km2": net_change,
            "Gain_Pixels": np.sum(gain),
            "Loss_Pixels": np.sum(loss)
        })

        # ΔNDWI map
        plt.figure(figsize=(8,6))
        plt.imshow(delta, cmap='bwr', vmin=-0.5, vmax=0.5)
        plt.contour(binary_stack[i], colors='blue', linewidths=0.5, alpha=0.5)
        plt.contour(binary_stack[i-1], colors='red', linewidths=0.5, alpha=0.3)
        plt.title(f"{sensor} ΔNDWI: {year_prev} → {year_curr}")
        plt.colorbar(label="ΔNDWI")
        out_file = os.path.join(sensor_folder, f"{sensor}_ndwi_change_{year_prev}_{year_curr}.png")
        plt.savefig(out_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✓ Year-to-year ΔNDWI map saved: {out_file}")

        # Per-pixel % change map
        with np.errstate(divide='ignore', invalid='ignore'):
            # delta_percent = ((ndwi_stack[i] - ndwi_stack[i-1]) / ndwi_stack[i-1]) * 100
            prev = ndwi_stack[i-1]      # or ndwi_stack[0] for earliest→latest
            curr = ndwi_stack[i]         # or ndwi_stack[-1] for earliest→latest
            mask = prev != 0              # only compute where previous year is not zero
            delta_percent = np.zeros_like(curr, dtype=float)
            delta_percent[mask] = ((curr[mask] - prev[mask]) / prev[mask]) * 100
            delta_percent[~mask] = 0      # or np.nan if you want to ignore these pixels
            delta_percent[np.isnan(delta_percent)] = 0
            delta_percent[np.isinf(delta_percent)] = 0
        plt.figure(figsize=(8,6))
        plt.imshow(delta_percent, cmap='RdBu', vmin=-100, vmax=100)
        plt.colorbar(label="% NDWI Change")
        plt.title(f"{sensor} % NDWI Change: {year_prev} → {year_curr}")
        out_file = os.path.join(sensor_folder, f"{sensor}_ndwi_percent_change_{year_prev}_{year_curr}.png")
        plt.savefig(out_file, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✓ Per-pixel % change map for {year_prev}->{year_curr} saved: {out_file}")

    # -----------------------------
    # Earliest → Latest Year ΔNDWI
    # -----------------------------
    delta_full = ndwi_stack[-1] - ndwi_stack[0]
    gain_full = delta_full > delta_threshold
    loss_full = delta_full < -delta_threshold
    area_gain_full = np.sum(gain_full) * pixel_area_km2
    area_loss_full = np.sum(loss_full) * pixel_area_km2
    net_change_full = area_gain_full - area_loss_full

    summary_rows.append({
        "Sensor": sensor,
        "Year_Prev": years_available[0],
        "Year_Curr": years_available[-1],
        "Area_Gained_km2": area_gain_full,
        "Area_Lost_km2": area_loss_full,
        "Net_Change_km2": net_change_full,
        "Gain_Pixels": np.sum(gain_full),
        "Loss_Pixels": np.sum(loss_full)
    })

    # ΔNDWI full period map
    plt.figure(figsize=(8,6))
    plt.imshow(delta_full, cmap='bwr', vmin=-0.5, vmax=0.5)
    plt.title(f"{sensor} ΔNDWI: {years_available[0]} → {years_available[-1]}")
    plt.colorbar(label="ΔNDWI")
    out_file = os.path.join(sensor_folder, f"{sensor}_ndwi_change_{years_available[0]}_{years_available[-1]}.png")
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Earliest → latest ΔNDWI map saved: {out_file}")

    # Per-pixel % change earliest → latest
    with np.errstate(divide='ignore', invalid='ignore'):
        delta_percent_full = ((ndwi_stack[-1] - ndwi_stack[0]) / ndwi_stack[0]) * 100
        delta_percent_full[np.isnan(delta_percent_full)] = 0
        delta_percent_full[np.isinf(delta_percent_full)] = 0
    plt.figure(figsize=(8,6))
    plt.imshow(delta_percent_full, cmap='RdBu', vmin=-100, vmax=100)
    plt.colorbar(label="% NDWI Change")
    plt.title(f"{sensor} % NDWI Change: {years_available[0]} → {years_available[-1]}")
    out_file = os.path.join(sensor_folder, f"{sensor}_ndwi_percent_change_{years_available[0]}_{years_available[-1]}.png")
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Per-pixel % earliest→latest change map saved: {out_file}")

    # -----------------------------
    # Sensor-specific multi-panel summary
    # -----------------------------
    n_years = len(years_available)
    fig, axes = plt.subplots(2, n_years-1, figsize=(4*(n_years-1), 8))  # landscape: 2 rows x N columns
    for i in range(1, n_years):
        delta = ndwi_stack[i] - ndwi_stack[i-1]

        # ΔNDWI
        axes[0, i-1].imshow(delta, cmap='bwr', vmin=-0.5, vmax=0.5)
        axes[0, i-1].set_title(f"ΔNDWI: {years_available[i-1]}→{years_available[i]}")
        axes[0, i-1].axis('off')

        # % change
        prev = ndwi_stack[i-1]
        curr = ndwi_stack[i]
        mask = prev != 0
        delta_percent = np.zeros_like(curr, dtype=float)
        delta_percent[mask] = ((curr[mask] - prev[mask]) / prev[mask]) * 100
        delta_percent[~mask] = 0
        delta_percent[np.isnan(delta_percent)] = 0
        delta_percent[np.isinf(delta_percent)] = 0

        axes[1, i-1].imshow(delta_percent, cmap='RdBu', vmin=-100, vmax=100)
        axes[1, i-1].set_title(f"% NDWI: {years_available[i-1]}→{years_available[i]}")
        axes[1, i-1].axis('off')

    plt.tight_layout()
    out_file = os.path.join(sensor_folder, f"{sensor}_multi_panel_summary.png")
    plt.savefig(out_file, dpi=300)
    plt.close()
    print(f"✓ Sensor multi-panel summary saved: {out_file}")


=== Processing sensor: landsat8 ===
landsat8 NDWI threshold suggestions:
  Median NDWI = -0.53
  5th percentile (wet pixels) = -0.67
  95th percentile = 1.00
✓ Multi-year water persistence map saved: ../data/results/landsat8/landsat8_water_persistence.png
✓ Year-to-year ΔNDWI map saved: ../data/results/landsat8/landsat8_ndwi_change_2016_2017.png
✓ Per-pixel % change map for 2016->2017 saved: ../data/results/landsat8/landsat8_ndwi_percent_change_2016_2017.png
✓ Year-to-year ΔNDWI map saved: ../data/results/landsat8/landsat8_ndwi_change_2017_2018.png
✓ Per-pixel % change map for 2017->2018 saved: ../data/results/landsat8/landsat8_ndwi_percent_change_2017_2018.png
✓ Year-to-year ΔNDWI map saved: ../data/results/landsat8/landsat8_ndwi_change_2018_2019.png
✓ Per-pixel % change map for 2018->2019 saved: ../data/results/landsat8/landsat8_ndwi_percent_change_2018_2019.png
✓ Year-to-year ΔNDWI map saved: ../data/results/landsat8/landsat8_ndwi_change_2019_2020.png
✓ Per-pixel % change map for 2

In [9]:
# -----------------------------
# Save summary CSV
# -----------------------------
summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(OUTPUT_DIR, "surface_water_change_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print(f"\n ✓ Quantitative summary CSV saved: {summary_csv}")


 ✓ Quantitative summary CSV saved: ../data/results/surface_water_change_summary.csv
